In [1]:
# import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations
import time

from itertools import product
import numpy as np
from multiprocessing import Pool
import os

import time

In [2]:
def cal_pow(R, beta, h_val):
    M = len(R)
    order = np.argsort(np.array(beta) / np.array(h_val))
    P = np.zeros(M)
    for i in range(M):
        ui = order[i]
        sum_i_M = sum(R[order[k]] for k in range(i, M))
        sum_ip1_M = sum(R[order[k]] for k in range(i+1, M))
        P[ui] = (2**sum_i_M - 2**sum_ip1_M) / h_val[ui]
    return P

def is_duplicate(c, C, eps=1e-6):
    if len(C) == 0:
        return True 
    else:
        flag = 0
        for c_old in C:
            if np.max(np.abs(c - c_old)) < eps:
                flag = 1
        if flag:
            return False
        else:
            return True

In [3]:
n_user = 3
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
rho = np.arange(0, r_max+1, 1)
####################################################
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
#####################################################
print('h:', h)
print('rho:', rho)

h: [[0.1, 0.1, 0.1], [0.1, 0.1, 1.0], [0.1, 1.0, 0.1], [0.1, 1.0, 1.0], [1.0, 0.1, 0.1], [1.0, 0.1, 1.0], [1.0, 1.0, 0.1], [1.0, 1.0, 1.0]]
rho: [0 1]


In [4]:


t1 = time.time()

def process_channel_k(args):
    k, h_k, hp_k, W, lambda_k, v_k, n_user = args
    costs = np.zeros(len(W))
    for j in range(len(W)):
        pow_j = cal_pow(W[j], lambda_k, h_k)
        for n in range(n_user):
            psi = 1 if W[j][n] > 0 else 0
            costs[j] += hp_k * (lambda_k[n] * pow_j[n] - v_k[n] * psi)

    min_cost = np.min(costs)
    idx      = np.where(np.abs(costs - min_cost) < 1e-8)[0]
    mu_k     = np.zeros(len(W))
    mu_k[idx] = 1.0 / len(idx)

    p_k  = np.zeros(n_user)
    Pu_k = np.zeros(n_user)
    for j in range(len(W)):
        pow_j = cal_pow(W[j], lambda_k, h_k)
        for n in range(n_user):
            if W[j][n] != 0:
                p_k[n]  += mu_k[j] * hp_k
            Pu_k[n] += pow_j[n] * mu_k[j] * hp_k

    return p_k, Pu_k


lambda_k  = n_user * [0.1]
v_k       = n_user * [0.1]
alpha     = 0.05
max_iter  = 10000

W         = list(product(rho, repeat=n_user))
acc_p     = np.zeros(n_user)
acc_Pu    = np.zeros(n_user)
acc_count = 0

n_workers = os.cpu_count()   # or set manually e.g. 4
print(f"Using {n_workers} workers")

with Pool(processes=n_workers) as pool:
    for itr in range(max_iter):
        if itr % 1000 == 0:
            print(itr)

        P_bar_U = n_user * [3]

        args = [(k, h[k], hp[k], W, lambda_k, v_k, n_user)
                for k in range(len(h))]

        results = pool.map(process_channel_k, args)

        p   = np.zeros(n_user)
        P_u = np.zeros(n_user)
        for p_k, Pu_k in results:
            p   += p_k
            P_u += Pu_k

        if itr > 5000:
            acc_p     += p
            acc_Pu    += P_u
            acc_count += 1

        subgradient_lambda = [P_u[n] - P_bar_U[n] for n in range(n_user)]
        subgradient_vk     = [1/np.sqrt(v_k[n]) - p[n] for n in range(n_user)]
        eta_k = alpha / np.sqrt(itr + 1)
        for n in range(n_user):
            lambda_k[n] = max(0, lambda_k[n] + eta_k * subgradient_lambda[n])
            v_k[n]      = max(0, v_k[n]      + eta_k * subgradient_vk[n])

print('--------------------------------------------------')
print('ERGODIC RESULTS')
p_er = acc_p / acc_count
age_bar = sum(1 / p_er[n] - 1 for n in range(n_user))
print("Final average age from ergodic mu:", np.round(age_bar, 5))

print('run time:', time.time() - t1)

Using 12 workers
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
--------------------------------------------------
ERGODIC RESULTS
Final average age from ergodic mu: 1.29201
run time: 9.698789119720459


In [ ]:
# import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations

from multiprocessing import Pool
import os
import time


def cal_pow(R, beta, h_val):
    M = len(R)
    order = np.argsort(np.array(beta) / np.array(h_val))
    P = np.zeros(M)
    for i in range(M):
        ui = order[i]
        sum_i_M = sum(R[order[k]] for k in range(i, M))
        sum_ip1_M = sum(R[order[k]] for k in range(i+1, M))
        P[ui] = (2**sum_i_M - 2**sum_ip1_M) / h_val[ui]
    return P

def is_duplicate(c, C, eps=1e-6):
    if len(C) == 0:
        return True 
    else:
        flag = 0
        for c_old in C:
            if np.max(np.abs(c - c_old)) < eps:
                flag = 1
        if flag:
            return False
        else:
            return True

n_user = 10
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
rho = np.arange(0, r_max+1, 1)
####################################################
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
#####################################################



t1 = time.time()

def process_channel_k(args):
    k, h_k, hp_k, W, lambda_k, v_k, n_user = args
    costs = np.zeros(len(W))
    for j in range(len(W)):
        pow_j = cal_pow(W[j], lambda_k, h_k)
        for n in range(n_user):
            psi = 1 if W[j][n] > 0 else 0
            costs[j] += hp_k * (lambda_k[n] * pow_j[n] - v_k[n] * psi)

    min_cost = np.min(costs)
    idx      = np.where(np.abs(costs - min_cost) < 1e-8)[0]
    mu_k     = np.zeros(len(W))
    mu_k[idx] = 1.0 / len(idx)

    p_k  = np.zeros(n_user)
    Pu_k = np.zeros(n_user)
    for j in range(len(W)):
        pow_j = cal_pow(W[j], lambda_k, h_k)
        for n in range(n_user):
            if W[j][n] != 0:
                p_k[n]  += mu_k[j] * hp_k
            Pu_k[n] += pow_j[n] * mu_k[j] * hp_k

    return p_k, Pu_k


lambda_k  = n_user * [0.1]
v_k       = n_user * [0.1]
alpha     = 0.05
max_iter  = 10000

W         = list(product(rho, repeat=n_user))
acc_p     = np.zeros(n_user)
acc_Pu    = np.zeros(n_user)
acc_count = 0

n_workers = os.cpu_count()   # or set manually e.g. 4
print(f"Using {n_workers} workers")

with Pool(processes=n_workers) as pool:
    for itr in range(max_iter):
       

        P_bar_U = n_user * [5]

        args = [(k, h[k], hp[k], W, lambda_k, v_k, n_user)
                for k in range(len(h))]

        results = pool.map(process_channel_k, args)

        p   = np.zeros(n_user)
        P_u = np.zeros(n_user)
        for p_k, Pu_k in results:
            p   += p_k
            P_u += Pu_k

        if itr > 10:
            acc_p     += p
            acc_Pu    += P_u
            acc_count += 1

        subgradient_lambda = [P_u[n] - P_bar_U[n] for n in range(n_user)]
        subgradient_vk     = [1/np.sqrt(v_k[n]) - p[n] for n in range(n_user)]
        eta_k = alpha / np.sqrt(itr + 1)
        for n in range(n_user):
            lambda_k[n] = max(0, lambda_k[n] + eta_k * subgradient_lambda[n])
            v_k[n]      = max(0, v_k[n]      + eta_k * subgradient_vk[n])

        if itr % 10 == 0:
            print(itr)
            p_er = acc_p / acc_count
            print(p_er)
            age_bar = sum(1 / p_er[n] - 1 for n in range(n_user))
            print("Final average age from ergodic mu:", np.round(age_bar, 5))

print('--------------------------------------------------')
print('ERGODIC RESULTS')
p_er = acc_p / acc_count
age_bar = sum(1 / p_er[n] - 1 for n in range(n_user))
print("Final average age from ergodic mu:", np.round(age_bar, 5))

print('run time:', time.time() - t1)


In [ ]:
import numpy as np
from itertools import product
from multiprocessing import Pool, cpu_count
import time

n_user = 10
h_bad = 0.1
h_good = 1.0
pr_h_bad = 0.5
pr_h_good = 1.0 - pr_h_bad
r_max = 1
rho = np.arange(0, r_max + 1)
P_bar_U = np.full(n_user, 3.0)
alpha = 0.05
max_iter = 10000


h_values = [h_bad, h_good]
h = np.array(
    list(product(h_values, repeat=n_user)),
    dtype=np.float64)
num_h = len(h)
prob_values = [pr_h_bad, pr_h_good]
hp = np.array(
    [
        np.prod(p)
        for p in product(prob_values, repeat=n_user)
    ],
    dtype=np.float64)

W = np.array(
    list(product(rho, repeat=n_user)),
    dtype=np.float64)

num_W = len(W)
psi = (W > 0).astype(np.float64)



def cal_pow_all(W, beta, h_val):
    M = len(beta)
    order = np.argsort(beta / h_val)
    W_ordered = W[:, order]
    rev_cumsum = np.cumsum(
        W_ordered[:, ::-1],
        axis=1
    )[:, ::-1]
    P = np.zeros_like(W)
    for i in range(M):
        ui = order[i]
        sum_i_M = rev_cumsum[:, i]
        if i == M - 1:
            sum_ip1_M = 0
        else:
            sum_ip1_M = rev_cumsum[:, i + 1]
        P[:, ui] = (
            2.0 ** sum_i_M
            - 2.0 ** sum_ip1_M
        ) / h_val[ui]
    return P


def process_channel_k(args):
    k, lambda_k, v_k = args
    h_k = h[k]
    Pk = cal_pow_all(W, lambda_k, h_k )

    costs = hp[k] * np.sum(lambda_k * Pk - v_k * psi, axis=1 )

    min_cost = np.min(costs)

    idx = np.where(np.abs(costs - min_cost) < 1e-10 )[0]

    mu = np.zeros(num_W)

    mu[idx] = 1.0 / len(idx)

    p_k = hp[k] * np.sum( mu[:, None] * psi, axis=0 )

    Pu_k = hp[k] * np.sum( mu[:, None] * Pk, axis=0 )

    return p_k, Pu_k


lambda_k = np.full(n_user, 0.1)
v_k = np.full(n_user, 0.1)

acc_p = np.zeros(n_user)
acc_Pu = np.zeros(n_user)

acc_count = 0

n_workers = cpu_count()

print(f"Using {n_workers} workers")

t1 = time.time()

with Pool(processes=n_workers) as pool:

    for itr in range(max_iter):

        args = [( k,lambda_k.copy(), v_k.copy()) for k in range(num_h)]

        results = pool.map( process_channel_k, args )

        p = np.zeros(n_user)
        P_u = np.zeros(n_user)

        for p_k, Pu_k in results:

            p += p_k
            P_u += Pu_k

        if itr > 100:

            acc_p += p
            acc_Pu += P_u

            acc_count += 1

        eta = alpha / np.sqrt(itr + 1)

        lambda_k = np.maximum(  0.0,  lambda_k + eta * ( P_u - P_bar_U ) )

        v_k = np.maximum(1e-10, v_k + eta * (1.0 / np.sqrt(v_k) - p ) )

        if itr % 100 == 0:
            print(f"Iteration = {itr}")
            p_er = acc_p / acc_count
            print(p_er)

            age_bar = np.sum(1.0 / p_er - 1.0 )

            print("Final average age:")
            print(np.round(age_bar, 6))
            print()


runtime = time.time() - t1

print()
print("==================================================")
print("ERGODIC RESULTS")
print("==================================================")

p_er = acc_p / acc_count


age_bar = np.sum( 1.0 / p_er - 1.0)

print("Final average age:")
print(np.round(age_bar, 6))
print()

print("Runtime:")
print(np.round(runtime, 3), "seconds")